<a href="https://colab.research.google.com/github/viktoria1user/Math_modelling/blob/main/%D0%A0%D0%B0%D1%81%D1%87%D0%B5%D1%82.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import numpy as np
from scipy.special import ellipk, ellipe
from numpy.polynomial.legendre import leggauss
import time

MU0 = 4 * np.pi * 1e-7
R = 0.1
d = 0.05

# Аналитическое значение
def neumann_analytical(R1, R2, d):
    k2 = 4 * R1 * R2 / ((R1 + R2)**2 + d**2)
    k = np.sqrt(k2)
    K = ellipk(k2)
    E = ellipe(k2)
    return MU0 * np.sqrt(R1 * R2) * ((2/k - k) * K - (2/k) * E)

M_analytic = neumann_analytical(R, R, d)
print(f"Аналитическое значение M = {M_analytic*1e6:.6f} мкГн\n")

# Метод средних прямоугольников
def neumann_midpoint(N):
    t = np.linspace(0, 2*np.pi, N, endpoint=False)
    dt = 2 * np.pi / N
    cos = np.cos(t)
    sin = np.sin(t)
    dl = np.stack([-R*sin, R*cos, np.zeros(N)], axis=1) * dt
    r1 = np.stack([R*cos, R*sin, np.zeros(N)], axis=1)
    r2 = r1.copy()
    r2[:, 2] = d

    M = 0.0
    for i in range(N):
        diff = r2 - r1[i]
        dist = np.linalg.norm(diff, axis=1)
        M += np.sum((dl[i] @ dl.T) / dist)
    return MU0 / (4*np.pi) * M

# Гаусс для кругов (специализированный)
def neumann_gauss_circles(nq):
    x, w = leggauss(nq)
    phi = np.pi * (x + 1.0)
    ww = np.pi * w
    phi1, phi2 = np.meshgrid(phi, phi, indexing='ij')
    W = np.outer(ww, ww)
    denom = np.sqrt(R**2 + R**2 + d**2 - 2*R*R*np.cos(phi1 - phi2))
    integrand = (R * R * np.cos(phi1 - phi2)) / denom
    return MU0 / (4*np.pi) * np.sum(W * integrand)

# Генерация таблицы
print("N     | Прямоугольники (%)   | Гаусс (%)")
print("-" * 50)

N_values = [5, 8, 10, 12, 16, 20, 24, 32, 48, 50, 64, 100]

for N in N_values:
    M_rect = neumann_midpoint(N)
    M_gauss = neumann_gauss_circles(N)

    err_rect = abs(M_rect - M_analytic) / abs(M_analytic) * 100
    err_gauss = abs(M_gauss - M_analytic) / abs(M_analytic) * 100

    print(f"{N:2d}    | {err_rect:15.10f}       | {err_gauss:14.10f}")

Аналитическое значение M = 0.111261 мкГн

N     | Прямоугольники (%)   | Гаусс (%)
--------------------------------------------------
 5    |   17.8797125206       |  29.5448480528
 8    |    3.0350879661       |   7.6526912221
10    |    0.9991696610       |   3.3792732673
12    |    0.3377338185       |   1.5469050972
16    |    0.0403163767       |   0.3456223999
20    |    0.0049788311       |   0.0813016528
24    |    0.0006277044       |   0.0197433791
32    |    0.0000103697       |   0.0012368364
48    |    0.0000000031       |   0.0000055713
50    |    0.0000000011       |   0.0000028593
64    |    0.0000000000       |   0.0000000277
100    |    0.0000000000       |   0.0000000000


In [19]:
import numpy as np
from scipy.special import ellipk, ellipe

MU0 = 4 * np.pi * 1e-7
R = 0.1
d = 0.05

# 1. Аналитическое значение (эталон)
def neumann_analytical(R1, R2, d):
    k2 = 4 * R1 * R2 / ((R1 + R2)**2 + d**2)
    k = np.sqrt(k2)
    K = ellipk(k2)
    E = ellipe(k2)
    return MU0 * np.sqrt(R1 * R2) * ((2/k - k) * K - (2/k) * E)

M_analytic = neumann_analytical(R, R, d)
print(f"Аналитическое значение: M = {M_analytic*1e6:.6f} мкГн\n")


# 2. Численный метод (средних прямоугольников)
def neumann_midpoint(N=500):
    t = np.linspace(0, 2*np.pi, N, endpoint=False)
    dt = 2 * np.pi / N
    cos = np.cos(t)
    sin = np.sin(t)
    # dl и r для первого контура
    dl1 = np.stack([-R*sin, R*cos, np.zeros(N)], axis=1) * dt
    r1 = np.stack([R*cos, R*sin, np.zeros(N)], axis=1)
    # второй контур
    r2 = r1.copy()
    r2[:, 2] = d
    dl2 = dl1.copy()
    M = 0.0
    for i in range(N):
        diff = r2 - r1[i]
        dist = np.linalg.norm(diff, axis=1)
        dot = dl1[i] @ dl2.T
        M += np.sum(dot / dist)

    return MU0 / (4 * np.pi) * M


# Запуск верификации
N = 500
M_num = neumann_midpoint(N)
err = abs(M_num - M_analytic) / abs(M_analytic) * 100
print(f"Численный результат (N = {N}):")
print(f"M = {M_num*1e6:.6f} мкГн")
print(f"Относительная погрешность: δ = {err:.4f} %")

Аналитическое значение: M = 0.111261 мкГн

Численный результат (N = 500):
M = 0.111261 мкГн
Относительная погрешность: δ = 0.0000 %
